In [1]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())

Current working directory: C:\Users\jarvis\Desktop\NextWord-AI\notebooks


In [2]:
PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)
print("Folders:", [p.name for p in PROJECT_ROOT.iterdir()])

Project root: C:\Users\jarvis\Desktop\NextWord-AI
Folders: ['.gitignore', 'app.py', 'data', 'models', 'notebooks', 'README.md', 'requirements.txt', 'src']


In [3]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "books"

ebook_path = RAW_DATA_PATH / "pride_prejudice.txt"

print("Dataset exists:", ebook_path.exists())

if ebook_path.exists():
    print("File size:", round(ebook_path.stat().st_size / 1024, 2), "KB")

Dataset exists: False


In [4]:
%pip install requests pandas numpy matplotlib seaborn nltk tensorflow

Note: you may need to restart the kernel to use updated packages.


In [5]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Current directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Git initialized:", (PROJECT_ROOT / ".git").exists())

Current directory: C:\Users\jarvis\Desktop\NextWord-AI\notebooks
Project root: C:\Users\jarvis\Desktop\NextWord-AI
Git initialized: False


In [7]:
!git -C .. init

Initialized empty Git repository in C:/Users/jarvis/Desktop/NextWord-AI/.git/


In [8]:
!git -C .. status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	app.py
	data/
	notebooks/
	requirements.txt
	src/

nothing added to commit but untracked files present (use "git add" to track)


In [9]:
!git -C .. branch -M main

In [10]:
!git -C .. add .
!git -C .. status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   app.py
	new file:   data/interim/.gitkeep
	new file:   data/processed/.gitkeep
	new file:   data/raw/.gitkeep
	new file:   notebooks/01_data_collection.ipynb
	new file:   notebooks/02_data_cleaning.ipynb
	new file:   notebooks/03_eda.ipynb
	new file:   notebooks/04_preprocessing.ipynb
	new file:   notebooks/05_model_training.ipynb
	new file:   requirements.txt
	new file:   src/cleaning.py
	new file:   src/model.py
	new file:   src/prediction.py
	new file:   src/preprocessing.py



In [11]:
!git -C .. commit -m "chore: initialize NextWord-AI project"

[main (root-commit) ecaaca5] chore: initialize NextWord-AI project
 16 files changed, 307 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 app.py
 create mode 100644 data/interim/.gitkeep
 create mode 100644 data/processed/.gitkeep
 create mode 100644 data/raw/.gitkeep
 create mode 100644 notebooks/01_data_collection.ipynb
 create mode 100644 notebooks/02_data_cleaning.ipynb
 create mode 100644 notebooks/03_eda.ipynb
 create mode 100644 notebooks/04_preprocessing.ipynb
 create mode 100644 notebooks/05_model_training.ipynb
 create mode 100644 requirements.txt
 create mode 100644 src/cleaning.py
 create mode 100644 src/model.py
 create mode 100644 src/prediction.py
 create mode 100644 src/preprocessing.py


In [25]:
!git -C .. remote set-url origin https://github.com/triyanjha04-dev/NextWord-AI.git

In [26]:
!git -C .. remote -v

origin	https://github.com/triyanjha04-dev/NextWord-AI.git (fetch)
origin	https://github.com/triyanjha04-dev/NextWord-AI.git (push)


In [27]:
!git -C .. push -u origin main

To https://github.com/triyanjha04-dev/NextWord-AI.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/triyanjha04-dev/NextWord-AI.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [28]:
!git -C .. fetch origin

From https://github.com/triyanjha04-dev/NextWord-AI
 * [new branch]      main       -> origin/main


In [29]:
!git -C .. pull origin main --allow-unrelated-histories

Merge made by the 'ort' strategy.
 .../extracting data-checkpoint.ipynb               |     6 +
 extracting data.ipynb                              |  1007 ++
 pride_prejudice.txt                                | 14915 +++++++++++++++++++
 3 files changed, 15928 insertions(+)
 create mode 100644 .ipynb_checkpoints/extracting data-checkpoint.ipynb
 create mode 100644 extracting data.ipynb
 create mode 100644 pride_prejudice.txt


From https://github.com/triyanjha04-dev/NextWord-AI
 * branch            main       -> FETCH_HEAD


In [30]:
!git -C .. status

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_data_collection.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/.~01_data_collection.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
!git -C .. push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/triyanjha04-dev/NextWord-AI.git
   d0a01d9..2ca9021  main -> main


In [32]:
import os
import requests
from pathlib import Path
import re
import html
import unicodedata

In [33]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
BOOKS_PATH = RAW_DATA_PATH / "books"

BOOKS_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Books folder:", BOOKS_PATH)

Project root: C:\Users\jarvis\Desktop\NextWord-AI
Books folder: C:\Users\jarvis\Desktop\NextWord-AI\data\raw\books


In [34]:
url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"

response = requests.get(url, timeout=30)

if response.status_code == 200:
    raw_text = response.text
    print("Download successful!")
    print("Total characters:", len(raw_text))
else:
    raise Exception(f"Download failed: {response.status_code}")

Download successful!
Total characters: 763082


In [35]:
raw_file_path = BOOKS_PATH / "pride_prejudice.txt"

raw_file_path.write_text(raw_text, encoding="utf-8")

print("Saved at:", raw_file_path)
print("File size:", round(raw_file_path.stat().st_size / 1024, 2), "KB")

Saved at: C:\Users\jarvis\Desktop\NextWord-AI\data\raw\books\pride_prejudice.txt
File size: 768.85 KB


In [37]:
print(raw_text[:2000])

The Project Gutenberg eBook of Pride and Prejudice
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Pride and Prejudice

Author: Jane Austen


        
Release date: June 1, 1998 [eBook #1342]
                Most recently updated: September 1, 2026

Language: English

Other information and formats: www.gutenberg.org/ebooks/1342

Credits: Chuck Greif and the Online Distributed Proofreading Team at http://www.pgdp.net (This file was produced from images available at The Internet Archive)


*** START OF THE PROJECT GUTENBERG EBOOK PRIDE AND PREJUDICE ***




                            [I